# Simple Local RAG System

**RAG = Retrieval-Augmented Generation**

Instead of relying only on what the AI memorized during training, RAG lets the AI **look up** information from *your own documents* before answering.

```
Your Question
     │
     ▼
[ Vector Search ] ──▶  Find the most relevant docs in your folder
     │
     ▼
[ LLM + Context ] ──▶  Generate an answer grounded in YOUR documents
```

### What we will build
| Step | What happens |
|------|--------------|
| 1 | Install libraries |
| 2 | Explore the `knowledge_base/` folder that comes with this repo |
| 3 | Convert documents into vectors and store them in ChromaDB |
| 4 | Retrieve the most relevant documents for a question |
| 5 | Generate an answer using Claude API + retrieved context |

### Libraries used
- **chromadb** – local vector database (stores and searches embeddings)
- **sentence-transformers** – converts text into vectors (embeddings)
- **anthropic** – Claude API for the generation step

---
## Step 1: Install Libraries

In [1]:
!pip install chromadb sentence-transformers groq python-dotenv

---
## Step 2: Explore the Knowledge Base

The `knowledge_base/` folder is already in this repo and contains four `.txt` files.
These are your documents — the AI will only answer based on what is in them.

In a real project you would replace these files with your own documents.

In [2]:
import os
from pathlib import Path

DOCS_FOLDER = "./knowledge_base"
os.makedirs(DOCS_FOLDER, exist_ok=True)

sample_documents = {
    "machine_learning.txt": """
Machine learning (ML) is a subset of artificial intelligence that allows computers
to learn from data without being explicitly programmed for every task.

Types of machine learning:
1. Supervised Learning  – the model learns from labeled examples (e.g. spam detection)
2. Unsupervised Learning – the model finds patterns in unlabeled data (e.g. clustering)
3. Reinforcement Learning – the model learns by trial and error using rewards

Key concepts:
- Training data: labeled examples used to teach the model
- Features: input variables the model uses to make predictions
- Loss function: measures how wrong the model's predictions are
- Gradient descent: algorithm used to minimize the loss and improve the model
""",
    "neural_networks.txt": """
Neural networks are computational models loosely inspired by the human brain.
They consist of layers of interconnected nodes called neurons.

Structure:
- Input layer  : receives the raw data
- Hidden layers: transform data through learned weights and activation functions
- Output layer : produces the final prediction

How they learn:
1. Forward pass      – data flows through the network to produce a prediction
2. Calculate loss    – compare prediction to the correct answer
3. Backpropagation   – calculate how much each weight contributed to the error
4. Update weights    – adjust weights using gradient descent

Common architectures:
- CNN  (Convolutional Neural Network) – great for images
- RNN  (Recurrent Neural Network)     – great for sequences and time-series
- Transformer                         – the architecture behind modern LLMs like Claude and GPT
""",
    "rag_explained.txt": """
RAG (Retrieval-Augmented Generation) is a technique that improves AI responses
by giving the model access to specific, up-to-date, or private information.

How RAG works:
1. Indexing   – documents are converted to vectors (embeddings) and stored in a vector DB
2. Retrieval  – a question is embedded and similar documents are retrieved via vector search
3. Generation – the retrieved documents are given to the LLM as context to answer the question

Why use RAG instead of just asking the LLM?
- Grounds the AI in YOUR specific data (company docs, research papers, code)
- Works with information that is newer than the model's training cutoff
- Reduces hallucinations by providing factual context
- No expensive fine-tuning required

What is a vector embedding?
A vector embedding is a list of numbers that captures the meaning of a piece of text.
Texts with similar meaning have vectors that are close together in space.
Cosine similarity is the most common way to measure how close two vectors are.
""",
    "python_tips.txt": """
Python is a high-level, interpreted programming language known for readability and simplicity.

Useful Python tips:
- Use list comprehensions instead of loops: [x * 2 for x in range(10)]
- Use f-strings for formatting: f"Hello, {name}!"
- Use virtual environments to isolate dependencies: python -m venv myenv
- Unpack tuples elegantly: first, *rest = [1, 2, 3, 4]
- Use enumerate() when you need both index and value: for i, v in enumerate(items)
- Use pathlib.Path instead of os.path for cleaner file handling

The Zen of Python (PEP 20) key principles:
- Beautiful is better than ugly
- Simple is better than complex
- Readability counts
- Errors should never pass silently
""",
}

for filename, content in sample_documents.items():
    Path(f"{DOCS_FOLDER}/{filename}").write_text(content.strip(), encoding="utf-8")

print(f"Created {len(sample_documents)} documents in '{DOCS_FOLDER}/'")
for name in sample_documents:
    print(f"  - {name}")

Created 4 documents in './knowledge_base/'
  - machine_learning.txt
  - neural_networks.txt
  - rag_explained.txt
  - python_tips.txt


---
## Step 3: Load Documents and Build the Vector Database

We do three things here:
1. **Load** all `.txt` files from the folder
2. **Embed** them – convert each document into a vector using a small local model
3. **Store** the vectors in ChromaDB so we can search them later

> The embedding model `all-MiniLM-L6-v2` (~80 MB) will download automatically on the first run.

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

# ── 1. Load documents from the folder ────────────────────────────────────────
def load_documents(folder_path: str) -> list[dict]:
    documents = []
    for filepath in Path(folder_path).glob("*.txt"):
        text = filepath.read_text(encoding="utf-8")
        documents.append({
            "id":     filepath.stem,   # filename without extension (used as unique ID)
            "text":   text,
            "source": filepath.name,
        })
    return documents

documents = load_documents(DOCS_FOLDER)
print(f"Loaded {len(documents)} documents")

# ── 2. Load the embedding model ───────────────────────────────────────────────
print("\nLoading embedding model (downloads ~80 MB on first run)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model ready!")

# ── 3. Create embeddings (one vector per document) ───────────────────────────
texts = [doc["text"]   for doc in documents]
ids   = [doc["id"]     for doc in documents]
metas = [{"source": doc["source"]} for doc in documents]

print("\nCreating embeddings...")
embeddings = embedding_model.encode(texts, show_progress_bar=True)
print(f"Each document is now a vector of {len(embeddings[0])} numbers")

# ── 4. Store in ChromaDB ──────────────────────────────────────────────────────
# PersistentClient saves the database to disk so it survives restarts
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Delete and recreate collection for a clean run
try:
    chroma_client.delete_collection("knowledge_base")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"},  # use cosine similarity for search
)

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metas,
)

print(f"\nStored {collection.count()} documents in ChromaDB (saved to ./chroma_db/)")

C:\Users\Tsvetomir.Teach\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 4 documents

Loading embedding model (downloads ~80 MB on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7461.89it/s]


Model ready!

Creating embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.10it/s]

Each document is now a vector of 384 numbers



Stored 4 documents in ChromaDB (saved to ./chroma_db/)


---
## Step 4: Retrieve Relevant Documents

Given a question, we:
1. Embed the question into a vector
2. Find the documents whose vectors are **closest** (most similar) to the question vector

This is the **Retrieval** part of RAG.

In [5]:
def retrieve(query: str, n_results: int = 2) -> list[dict]:
    """Find the most relevant documents for a given query."""
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved.append({
            "text":       doc,
            "source":     meta["source"],
            "similarity": round(1 - dist, 3),  # convert distance → similarity score
        })

    return retrieved


# ── Test retrieval ────────────────────────────────────────────────────────────
test_query = "What is machine learning?"
results = retrieve(test_query, n_results=2)

print(f'Query: "{test_query}"\n')
for i, result in enumerate(results, 1):
    print(f"{i}. Source: {result['source']}  |  Similarity: {result['similarity']}")
    print(f"   Preview: {result['text'][:120].strip()}...")
    print()

Query: "What is machine learning?"

1. Source: machine_learning.txt  |  Similarity: 0.76
   Preview: Machine learning (ML) is a subset of artificial intelligence that allows computers
to learn from data without being expl...

2. Source: neural_networks.txt  |  Similarity: 0.431
   Preview: Neural networks are computational models loosely inspired by the human brain.
They consist of layers of interconnected n...



---
## Step 5: Generate an Answer (RAG = Retrieval + Generation)

Now we combine retrieval with a language model:
1. Retrieve relevant documents
2. Pass them as **context** to Llama 3.1 8B (running on Groq)
3. The model answers using only the provided context

> **Before running this cell:** create a `.env` file in the same folder as this notebook with your Groq API key (free at [console.groq.com](https://console.groq.com)):
> ```
> GROQ_API_KEY=gsk_...
> ```
> The `.env` file is the recommended way to handle API keys in Jupyter on Windows — environment variables set via System Properties are often not visible to the kernel.

In [7]:
import os
import httpx
import warnings
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Corporate/school networks often use SSL inspection proxies with self-signed
# certificates, which causes SSL verification errors. This disables that check.
warnings.filterwarnings("ignore", message="Unverified HTTPS request")
groq_client = Groq(http_client=httpx.Client(verify=False))


def rag_answer(question: str, n_results: int = 2) -> str:
    """
    Full RAG pipeline:
      1. Retrieve the most relevant documents for the question
      2. Build a prompt with those documents as context
      3. Ask Llama 3.1 8B to answer using only the provided context
    """
    # Step 1 – Retrieve
    relevant_docs = retrieve(question, n_results=n_results)

    # Step 2 – Build context string
    context = "\n\n---\n\n".join(
        f"[Source: {doc['source']}]\n{doc['text']}"
        for doc in relevant_docs
    )

    # Step 3 – Generate
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=512,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. "
                    "Answer the user's question using ONLY the context provided below. "
                    "If the context does not contain enough information, say so clearly."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}",
            },
        ],
    )

    return response.choices[0].message.content


# ── Test the full RAG pipeline ────────────────────────────────────────────────
question = "What is machine learning and how does it relate to neural networks?"
print(f"Q: {question}\n")
answer = rag_answer(question)
print(f"A: {answer}")

Q: What is machine learning and how does it relate to neural networks?

A: Machine learning (ML) is a subset of artificial intelligence that allows computers to learn from data without being explicitly programmed for every task.

Machine learning is closely related to neural networks. In fact, neural networks are a key component in many machine learning algorithms. They are computational models that allow the model to learn and improve from data by adjusting the weights and connections between different nodes called neurons.

More specifically, neural networks can be used to implement different types of machine learning, such as:

* Supervised Learning, where the model learns from labeled examples (e.g., spam detection)
* Unsupervised Learning, where the model finds patterns in unlabeled data (e.g., clustering)
* Reinforcement Learning, where the model learns by trial and error using rewards

Within machine learning, neural networks are trained using algorithms like gradient descent an

---
## Step 6: Try Your Own Questions!

In [8]:
my_questions = [
    "What is RAG and why should I use it?",
    "How does backpropagation work?",
    "What are some tips for writing better Python code?",
    # Add your own questions here!
]

for q in my_questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print("-" * 70)

Q: What is RAG and why should I use it?
A: RAG, or Retrieval-Augmented Generation, is a technique that improves AI responses by giving the model access to specific, up-to-date, or private information. This is achieved through the following steps: 

1. Indexing - documents are converted to vectors (embeddings) and stored in a vector database.
2. Retrieval - a question is embedded and similar documents are retrieved via vector search.
3. Generation - the retrieved documents are given to the LLM as context to answer the question.

You should use RAG instead of just asking the LLM because:
- It grounds the AI in your specific data, such as company documents, research papers, or code.
- It works with information that is newer than the model's training cutoff.
- It reduces hallucinations by providing factual context.
- No expensive fine-tuning is required.
----------------------------------------------------------------------
Q: How does backpropagation work?
A: According to the context prov

---
## Summary

You just built a working RAG system with only ~50 lines of code!

```
knowledge_base/          ← your documents live here
    machine_learning.txt
    neural_networks.txt
    rag_explained.txt
    python_tips.txt

chroma_db/               ← ChromaDB stores the vectors here (persists on disk)
```

### Key concepts to remember

| Concept | What it does |
|---------|-------------|
| **Embedding** | Converts text into a vector (list of numbers) that captures meaning |
| **Vector DB** | Stores vectors and finds the closest ones to a query vector |
| **Cosine similarity** | Measures how "close" two vectors are (1 = identical, 0 = unrelated) |
| **Context window** | The retrieved documents passed to the LLM as extra information |

### What to try next
- **Point it at your own documents** – change `DOCS_FOLDER` to any folder on your computer
- **Chunk large documents** – split big files into smaller paragraphs before embedding
- **Add more file types** – load `.pdf` or `.docx` files with libraries like `pypdf` or `python-docx`
- **Tune retrieval** – experiment with `n_results` to see how more/fewer docs affect the answer